# MoneyLab — Financial Planning & Decision Simulator

**University Application Development Project · Phase 1 demonstration**

MoneyLab is a general-purpose Python application for modeling cash savings, monthly cash flow, one loan's amortization, and financial scenarios. It targets students, recent graduates, and early-career professionals, but uses no person-specific financial information.

The interactive application is in [the MoneyLab GitHub repository](https://github.com/jbj-02/moneylab). This notebook runs the **same calculation engine** used by the Streamlit application; it does not duplicate the financial formulas.

**Run instructions:** Select **Runtime → Run all**. Cells use fictional values and do not require credentials, paid APIs, or personal banking data.

## Problem, purpose, and features

Financial decisions often involve difficult tradeoffs: saving monthly surplus, repaying debts, and preparing for the future. A spreadsheet can provide calculations but is less convenient for exploring changing assumptions.

MoneyLab collects a basic financial profile, models interest and loan repayment month by month, and shows how savings, debt, and partial net financial position change. The first implementation supports income, expenses, savings, loan interest, scheduled repayments, and flexible horizons. Later features include comparing extra loan payments to saving.

**Model limitations:** Fixed income, expenses, and interest rates; one loan; no inflation, savings yield, investment returns, or other assets and debts. Shortfalls are tracked as unfunded amounts. Results are projections, not forecasts or financial advice.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Colab typically starts in /content. On a local checkout, use the checkout.
cwd = Path.cwd()
if (cwd / "moneylab" / "engine.py").is_file():
    project_root = cwd
elif (cwd.parent / "moneylab" / "engine.py").is_file():
    project_root = cwd.parent
else:
    project_root = cwd / "moneylab"
    if not (project_root / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/jbj-02/moneylab.git", str(project_root)],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(project_root), "pull", "--ff-only"],
            check=True,
        )

sys.path.insert(0, str(project_root))
print("Using MoneyLab source from:", project_root)

## 1. Run the monthly financial projection

These are **hypothetical inputs**. Living expenses explicitly exclude debt payments. The model calculates monthly loan interest (annual percentage rate divided by 12), caps final repayments to the amount owed, then updates the cash position. 

In [ ]:
from moneylab.engine import project_finances
import pandas as pd

inputs = {
    "current_savings": 5000,
    "monthly_income": 3000,
    "monthly_living_expenses": 2000,
    "outstanding_loan_balance": 12000,
    "annual_loan_interest_rate": 5,
    "required_monthly_loan_payment": 250,
    "months": 24,
}

projection = project_finances(**inputs)
summary = projection["summary"]
print("Ending cash savings:", summary["ending_savings"])
print("Remaining loan debt:", summary["ending_loan_balance"])
print("Ending modeled net position:", summary["ending_net_position"])
print("Total interest charged:", summary["total_interest_charged"])
print("Loan payoff month:", summary["loan_payoff_month"])

month_data = pd.DataFrame(projection["monthly_records"])
display(month_data[["month", "savings", "loan_balance", "net_position"]].head(7))

## 2. Visualize the financial trajectory

The chart starts at Month 0 and shows the balance changes for each modeled month.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for key, label in (
    ("savings", "Cash savings"),
    ("loan_balance", "Loan debt"),
    ("net_position", "Partial net position"),
):
    ax.plot(month_data["month"], month_data[key].astype(float), label=label)
ax.set_xlabel("Month")
ax.set_ylabel("USD")
ax.set_title("MoneyLab — Hypothetical 24-Month Projection")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## 3. Compare saving and paying extra toward a loan

Both scenarios use identical financial assumptions. Optional additional debt payments use at most the affordable monthly surplus, never the existing cash savings. This feature extends the original Phase 1 calculation engine.

In [ ]:
from moneylab.strategies import compare_financial_strategies

scenarios = compare_financial_strategies(
    **inputs,
    extra_monthly_loan_payment=200,
)

display(pd.DataFrame({
    "Measure": ["Ending savings", "Remaining debt", "Interest charged",
                "Partial net position", "Loan payoff month"],
    "Save surplus": [
        scenarios["saving"]["summary"][name] for name in (
            "ending_savings", "ending_loan_balance",
            "total_interest_charged", "ending_net_position", "loan_payoff_month"
        )
    ],
    "Pay extra toward debt": [
        scenarios["accelerated"]["summary"][name] for name in (
            "ending_savings", "ending_loan_balance",
            "total_interest_charged", "ending_net_position", "loan_payoff_month"
        )
    ],
}))

print("Interest avoided:", scenarios["comparison"]["interest_avoided"])

## 4. Quick financial integrity checks

These assertions establish that each month's change in cash equals net cash flow and each loan's balance reconciles with interest charged and payments. Comprehensive automated tests are in [the repository's tests folder](https://github.com/jbj-02/moneylab/tree/main/tests).

In [ ]:
from decimal import Decimal

for previous, current in zip(
    projection["monthly_records"], projection["monthly_records"][1:]
):
    assert current["cash_position"] == previous["cash_position"] + current["net_cash_flow"]
    assert current["loan_balance"] == (
        previous["loan_balance"] + current["interest_charged"] - current["loan_payment"]
    )
    assert current["net_position"] == (
        current["savings"] - current["unfunded_shortfall"] - current["loan_balance"]
    )
print("Financial integrity checks passed.")

## Running the full application

For the interactive dashboard, clone the repository locally, install the requirements from `requirements.txt`, then execute `python -m streamlit run app.py`.

This Colab notebook provides a reproducible code demonstration and should be saved/shared with the professor using their required notebook-sharing process.